# YoungNAKO · Machine Learning · Notebook 4: **For home**
#### Thomas Küstner, University Hospital of Tübingen
## Train it yourself & run it on your own NAKO Dixon data

This notebook is **not** part of the course time. It needs a GPU and patience (≥ 1 h for the training parts).

| Part | What | Runtime |
|---|---|---|
| **A** | Train an **nnU-Net** on CHAOS (in-phase + opposed-phase as 2 input channels) | 1–2 h (short schedule) to ~1 day (full) on one GPU |
| **B** | **Fine-tune BiomedCLIP** contrastively on slice–caption pairs | ~15 min on a GPU |
| **C** | Interactive / promptable segmentation: **MedSAM2, nnInteractive** (pointers) | n/a |
| **D** | **Your own NAKO Dixon data**: organ volumes & fat fractions with TotalSegmentator MRI → CSV → regression model | ~10 s per participant (GPU) |

> 🔒 **Data protection:** Part D is meant to run **locally** on infrastructure approved for your NAKO data (your institute's GPU server or workstation, not Google Colab). Nothing has to be uploaded or shared. All code runs offline once the packages and model weights are installed.

In [ ]:
# Install the requirements into the running kernel: requirements.txt of the repo if it is next to
# this notebook (local / cloned repo), otherwise (Colab opened via link) this notebook's packages.
import os
_req = next((p for p in ("requirements.txt", "../requirements.txt") if os.path.isfile(p)), None)
get_ipython().run_line_magic("pip", f"install -q -r {_req}" if _req else
                             "install -q numpy pandas matplotlib scipy scikit-learn torch pillow pydicom SimpleITK transformers nnunetv2 TotalSegmentator open_clip_torch statsmodels")
import os, sys, glob, subprocess, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEV = DEVICE                                              # used in the nnU-Net command lines
WORK = os.environ.get("YN_WORK", "/content/yn_work" if os.path.isdir("/content") else os.path.expanduser("~/yn_data/yn_work"))
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]   # so !nnUNetv2_* uses this kernel's install
os.makedirs(WORK, exist_ok=True)
print("Running on:", DEVICE, "| work dir:", WORK)

In [ ]:
#@title 🔧 Helper functions: download & load the CHAOS MRI data (just run this cell) { display-mode: "form" }
# CHAOS challenge data (Kavur et al., Med Image Anal 2021), Zenodo record 3431873, licence CC BY-NC-SA 4.0.
# We use the 20 MR training cases, T1-DUAL sequence: in-phase (IP) and opposed-phase (OP) images
# plus expert masks of liver, right kidney, left kidney and spleen.
import os, glob, zipfile, urllib.request, shutil
import numpy as np

CHAOS_URL = "https://zenodo.org/records/3431873/files/CHAOS_Train_Sets.zip?download=1"
# Colab: /content/chaos · locally: ~/yn_data/chaos (shared by all notebooks, outside Dropbox/git); override with YN_CHAOS_ROOT
CHAOS_DIR = os.environ.get("YN_CHAOS_ROOT", "/content/chaos" if os.path.isdir("/content") else os.path.expanduser("~/yn_data/chaos"))
LABELS = {1: "liver", 2: "kidney_right", 3: "kidney_left", 4: "spleen"}


def download_chaos(dest=CHAOS_DIR):
    """Download (≈ 0.9 GB) and unpack only the MR part of the CHAOS training set."""
    if glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True):
        print("CHAOS data already available in", dest)
        return dest
    os.makedirs(dest, exist_ok=True)
    zpath = os.path.join(dest, "CHAOS_Train_Sets.zip")
    if not os.path.exists(zpath):
        print("Downloading CHAOS training set from Zenodo (≈ 0.9 GB, 1-3 min on Colab) ...")
        with urllib.request.urlopen(CHAOS_URL) as r, open(zpath, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 22)
    with zipfile.ZipFile(zpath) as z:
        members = [m for m in z.namelist() if "/MR/" in m.replace("\\", "/") and "T1DUAL" in m]
        z.extractall(dest, members)
    os.remove(zpath)
    print("Done:", len(glob.glob(os.path.join(dest, "**", "T1DUAL"), recursive=True)), "MR cases")
    return dest


def _png_to_labels(png):
    """CHAOS ground-truth PNGs encode organs as grey values: liver 63, R kidney 126, L kidney 189, spleen 252."""
    lab = np.zeros(png.shape, np.uint8)
    lab[(png >= 55) & (png <= 70)] = 1
    lab[(png >= 110) & (png <= 135)] = 2
    lab[(png >= 175) & (png <= 200)] = 3
    lab[(png >= 240)] = 4
    return lab


def load_case(case_dir):
    """Load one CHAOS T1-DUAL case.
    Returns dict with 'ip', 'op' (float32, z×y×x), 'mask' (uint8, z×y×x),
    'spacing' (x, y, z in mm), 'origin', 'direction' (for building a 3D image), 'id'."""
    import pydicom
    from PIL import Image
    t1 = os.path.join(case_dir, "T1DUAL")
    ip_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "InPhase", "*.dcm")))
    op_files = sorted(glob.glob(os.path.join(t1, "DICOM_anon", "OutPhase", "*.dcm")))
    gt_files = sorted(glob.glob(os.path.join(t1, "Ground", "*.png")))
    n = min(len(ip_files), len(op_files), len(gt_files))
    if not (len(ip_files) == len(op_files) == len(gt_files)):
        print(f"  note: case {os.path.basename(case_dir)} has {len(ip_files)}/{len(op_files)}/{len(gt_files)} IP/OP/GT slices, using {n}")
    ip_ds = [pydicom.dcmread(f) for f in ip_files[:n]]
    op_ds = [pydicom.dcmread(f) for f in op_files[:n]]
    gt = [_png_to_labels(np.array(Image.open(f).convert("L"))) for f in gt_files[:n]]
    # geometry from DICOM header -> sort slices along the slice normal
    iop = np.array(ip_ds[0].ImageOrientationPatient, float)
    r, c = iop[:3], iop[3:]
    nrm = np.cross(r, c)
    pos = np.array([np.dot(np.array(d.ImagePositionPatient, float), nrm) for d in ip_ds])
    order = np.argsort(pos)
    ip = np.stack([ip_ds[i].pixel_array.astype(np.float32) for i in order])
    op = np.stack([op_ds[i].pixel_array.astype(np.float32) for i in order])
    mask = np.stack([gt[i] for i in order])
    dz = float(np.median(np.diff(pos[order]))) if n > 1 else float(ip_ds[0].SliceThickness)
    ps = [float(v) for v in ip_ds[0].PixelSpacing]  # [row spacing, column spacing]
    direction = (r[0], c[0], nrm[0], r[1], c[1], nrm[1], r[2], c[2], nrm[2])
    origin = tuple(float(v) for v in ip_ds[order[0]].ImagePositionPatient)
    return dict(ip=ip, op=op, mask=mask, spacing=(ps[1], ps[0], abs(dz)), origin=origin,
                direction=direction, id=os.path.basename(case_dir))


def load_all_cases(root=CHAOS_DIR):
    case_dirs = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "MR", "*", "T1DUAL"), recursive=True)},
                       key=lambda s: int(os.path.basename(s)) if os.path.basename(s).isdigit() else s)
    cases = [load_case(d) for d in case_dirs]
    print(f"Loaded {len(cases)} cases, {sum(c['ip'].shape[0] for c in cases)} slices per contrast")
    return cases


def to_sitk(vol, case):
    """Wrap a z×y×x numpy array into a SimpleITK image with the case geometry."""
    import SimpleITK as sitk
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing(case["spacing"]); img.SetOrigin(case["origin"]); img.SetDirection(case["direction"])
    return img


def _norm(vol):
    lo, hi = np.percentile(vol, 1), np.percentile(vol, 99.5)
    return np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)


def make_slice_dataset(cases, size=128, min_kidney_px=50):
    """Turn the 3D cases into a 2D slice dataset.
    X_ip, X_op : (N, size, size) images scaled to [0, 1] (per volume)
    y_kidney   : 1 if at least one kidney is visible in the slice
    z_rel      : relative slice position in the volume (0 = lowest slice, 1 = highest)
    patient    : case id (for patient-level splitting!)
    organs     : list of visible organs per slice (used for captions)"""
    import torch, torch.nn.functional as F
    X_ip, X_op, M, yk, zr, pid, organs = [], [], [], [], [], [], []
    for c in cases:
        ipn, opn = _norm(c["ip"]), _norm(c["op"])
        nz = ipn.shape[0]
        for z in range(nz):
            m = c["mask"][z]
            X_ip.append(ipn[z]); X_op.append(opn[z]); M.append(m)
            k = int(((m == 2) | (m == 3)).sum() >= min_kidney_px)
            yk.append(k); zr.append(z / max(nz - 1, 1)); pid.append(c["id"])
            organs.append([LABELS[l] for l in (1, 2, 3, 4) if (m == l).sum() >= min_kidney_px])
    def rs(a, mode):
        t = torch.from_numpy(np.stack(a)[:, None].astype(np.float32))
        t = F.interpolate(t, size=(size, size), mode=mode, **({} if mode == "nearest" else {"align_corners": False}))
        return t[:, 0].numpy()
    X_ip, X_op = rs(X_ip, "bilinear"), rs(X_op, "bilinear")
    M = rs(M, "nearest").astype(np.uint8)
    return dict(X_ip=X_ip, X_op=X_op, mask=M, y_kidney=np.array(yk), z_rel=np.array(zr, np.float32),
                patient=np.array(pid), organs=organs)


def patient_split(patients, n_val=3, n_test=4, seed=0):
    """Split by PATIENT (never by slice!) into train/val/test index arrays."""
    ids = np.array(sorted(set(patients), key=lambda s: int(s) if str(s).isdigit() else s))
    rng = np.random.default_rng(seed); rng.shuffle(ids)
    test_ids, val_ids, train_ids = ids[:n_test], ids[n_test:n_test + n_val], ids[n_test + n_val:]
    idx = lambda s: np.where(np.isin(patients, s))[0]
    return idx(train_ids), idx(val_ids), idx(test_ids)

In [ ]:
download_chaos()
cases = load_all_cases()

---
# Part A · Train your own nnU-Net on CHAOS

nnU-Net (Isensee et al., *Nature Methods* 2021) configures pre-processing, network and training **automatically** from a dataset fingerprint. You only bring the data in its folder format:

```
nnUNet_raw/Dataset501_CHAOS/
 ├─ imagesTr/  chaos_01_0000.nii.gz (in-phase)  chaos_01_0001.nii.gz (opposed-phase) ...
 ├─ labelsTr/  chaos_01.nii.gz  (0 background, 1 liver, 2 right kidney, 3 left kidney, 4 spleen)
 ├─ imagesTs/  held-out test patients
 └─ dataset.json
```

In [ ]:
import json, SimpleITK as sitk
os.environ["nnUNet_raw"] = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{WORK}/nnUNet_results"
DS = f"{os.environ['nnUNet_raw']}/Dataset501_CHAOS"
for d in ("imagesTr", "labelsTr", "imagesTs", "labelsTs"): os.makedirs(f"{DS}/{d}", exist_ok=True)

TEST_IDS = {c["id"] for c in cases[-4:]}               # keep 4 patients completely unseen
for c in cases:
    split = "Ts" if c["id"] in TEST_IDS else "Tr"
    name = f"chaos_{int(c['id']):02d}" if str(c["id"]).isdigit() else f"chaos_{c['id']}"
    sitk.WriteImage(to_sitk(c["ip"], c), f"{DS}/images{split}/{name}_0000.nii.gz")
    sitk.WriteImage(to_sitk(c["op"], c), f"{DS}/images{split}/{name}_0001.nii.gz")
    sitk.WriteImage(to_sitk(c["mask"], c), f"{DS}/labels{split}/{name}.nii.gz")

json.dump({"channel_names": {"0": "in_phase", "1": "opposed_phase"},      # non-CT names -> z-score normalisation
           "labels": {"background": 0, "liver": 1, "kidney_right": 2, "kidney_left": 3, "spleen": 4},
           "numTraining": len(glob.glob(f"{DS}/labelsTr/*.nii.gz")), "file_ending": ".nii.gz"},
          open(f"{DS}/dataset.json", "w"), indent=2)
print(open(f"{DS}/dataset.json").read())

In [ ]:
# 1) Fingerprint + experiment planning + pre-processing (a few minutes)
!nnUNetv2_plan_and_preprocess -d 501 -c 3d_fullres --verify_dataset_integrity

**Training schedule.** The full nnU-Net trains 1000 epochs × 5 folds (days). For a first experiment, use a shortened schedule from nnU-Net's built-in variants: `nnUNetTrainer_5epochs`, `_50epochs`, `_100epochs`, `_250epochs` … or the full `nnUNetTrainer`.

In [ ]:
TRAINER = "nnUNetTrainer_100epochs"     # full model: "nnUNetTrainer"
CONFIG = "3d_fullres"
FOLD = "0"
# 2) Train one fold (on a T4 GPU ≈ 1–2 min per epoch for this small dataset)
!nnUNetv2_train 501 {CONFIG} {FOLD} -tr {TRAINER} -device {DEV}

In [ ]:
# 3) Predict the 4 held-out patients and evaluate
PRED = f"{WORK}/pred_nnunet"
!nnUNetv2_predict -i {DS}/imagesTs -o {PRED} -d 501 -c {CONFIG} -f {FOLD} -tr {TRAINER} -device {DEV} --disable_tta

def dice(a, b): return 2 * (a & b).sum() / (a.sum() + b.sum() + 1e-9)
rows = []
for f in sorted(glob.glob(f"{PRED}/*.nii.gz")):
    p = sitk.GetArrayFromImage(sitk.ReadImage(f)); g = sitk.GetArrayFromImage(sitk.ReadImage(f"{DS}/labelsTs/{os.path.basename(f)}"))
    rows.append({"case": os.path.basename(f), **{n: dice(p == l, g == l) for l, n in LABELS.items()}})
pd.DataFrame(rows).round(3)

**Ideas to explore:** train with only the in-phase channel (delete `_0001` files and the channel in `dataset.json`) → does the opposed-phase contrast help? Compare with TotalSegmentator MRI (Notebook 2, no training at all).

---
# Part B · Fine-tune BiomedCLIP contrastively

We continue training the pre-trained BiomedCLIP on our own image–caption pairs (captions generated from the CHAOS masks, as in Notebook 2), only updating the last projection layers. This is the same contrastive loss as CLIP pre-training, just with *our* data.

In [ ]:
import open_clip, torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score
HF_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
clip_model, clip_pre = open_clip.create_model_from_pretrained(HF_ID)
clip_tok = open_clip.get_tokenizer(HF_ID)
clip_model = clip_model.to(DEVICE)

data = make_slice_dataset(cases, size=224)
def caption(phase, z, organs):
    region = "upper" if z > 0.66 else "middle" if z > 0.33 else "lower"
    org = " and ".join(o.replace("_", " ") for o in organs) if organs else "no labelled organ"
    return f"{phase} T1-weighted MRI of the {region} abdomen showing {org}"
X_all = np.concatenate([data["X_ip"], data["X_op"]])
caps = [caption("in-phase", z, o) for z, o in zip(data["z_rel"], data["organs"])] + \
       [caption("opposed-phase", z, o) for z, o in zip(data["z_rel"], data["organs"])]
pat2 = np.concatenate([data["patient"], data["patient"]]); yk2 = np.concatenate([data["y_kidney"]] * 2)
tr, va, te = patient_split(data["patient"], n_val=3, n_test=4, seed=0)
tr_mask = np.isin(pat2, list(set(data["patient"][tr]))); te_mask = np.isin(pat2, list(set(data["patient"][te])))
to_tensor = lambda A: torch.stack([clip_pre(Image.fromarray((x * 255).astype(np.uint8)).convert("RGB")) for x in A])

@torch.no_grad()
def zero_shot_auc(prompts=("axial MRI of the abdomen showing kidney right and kidney left",
                           "axial MRI of the upper abdomen showing liver and spleen, no kidney")):
    clip_model.eval()
    t = F.normalize(clip_model.encode_text(clip_tok(list(prompts), context_length=256).to(DEVICE)).float(), dim=-1)
    idx = np.where(te_mask)[0]; s = []
    for i in range(0, len(idx), 64):
        f = F.normalize(clip_model.encode_image(to_tensor(X_all[idx[i:i+64]]).to(DEVICE)).float(), dim=-1)
        s.append((100 * f @ t.T).softmax(-1)[:, 0].cpu())
    return roc_auc_score(yk2[idx], torch.cat(s).numpy())

print("zero-shot AUC BEFORE fine-tuning:", round(zero_shot_auc(), 3))

In [ ]:
# Freeze everything except projection heads and the temperature; unfreeze more layers if you have a bigger GPU
for n, p in clip_model.named_parameters():
    p.requires_grad = any(k in n for k in ("proj", "head", "logit_scale"))
params = [p for p in clip_model.parameters() if p.requires_grad]
print("trainable tensors:", len(params), "| parameters:", sum(p.numel() for p in params))
opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=0.01)
idx_tr = np.where(tr_mask)[0]
for ep in range(3):
    clip_model.train(); perm = np.random.permutation(idx_tr); tot = 0
    for i in range(0, len(perm), 32):
        b = perm[i:i + 32]
        fi = F.normalize(clip_model.encode_image(to_tensor(X_all[b]).to(DEVICE)), dim=-1)
        ft = F.normalize(clip_model.encode_text(clip_tok([caps[k] for k in b], context_length=256).to(DEVICE)), dim=-1)
        logits = clip_model.logit_scale.exp() * fi @ ft.T
        target = torch.arange(len(b), device=DEVICE)
        loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
    print(f"epoch {ep+1}: loss {tot/len(idx_tr):.3f} | zero-shot test AUC {zero_shot_auc():.3f}")

---
# Part C · Promptable & interactive segmentation (pointers)

| Tool | What it does | Where to start |
|---|---|---|
| **MedSAM2** (Ma et al., 2025) | "Segment anything" for 3D images and videos: draw a **box on one slice**, the mask is propagated through the volume | GitHub `bowang-lab/MedSAM2` (official Colab and 3D Slicer plugin linked in the README) |
| **nnInteractive** (Isensee et al., 2025) | 3D interactive segmentation with points, scribbles, boxes and lasso | napari plugin and 3D Slicer extension; GitHub `MIC-DKFZ/nnInteractive` |
| **MRSegmentator** (Häntze et al., 2024) | 40 structures in MRI and CT; trained on **NAKO** and UK Biobank Dixon data, among others | `pip install mrsegmentator` |
| **TotalSegmentator** `tissue_types_mr` | subcutaneous fat, torso fat, skeletal muscle | needs a free academic licence: `totalseg_set_license -l <key>` |

The cell below exports one CHAOS case and the liver bounding box, which is exactly the input MedSAM2 expects.

In [ ]:
c = cases[0]
sitk.WriteImage(to_sitk(c["ip"], c), f"{WORK}/chaos_{c['id']}_ip.nii.gz")
z_mid = int(np.argmax((c["mask"] == 1).sum((1, 2))))
ys, xs = np.nonzero(c["mask"][z_mid] == 1)
print(f"file: {WORK}/chaos_{c['id']}_ip.nii.gz")
print(f"liver box on slice z={z_mid}: x0={xs.min()}, y0={ys.min()}, x1={xs.max()}, y1={ys.max()}  (array coordinates)")

---
# Part D · Your own NAKO Dixon data → image-derived phenotypes

NAKO's 3D Dixon (VIBE) gives four images per participant: **in-phase, opposed-phase, fat, water**. The pipeline:

1. read the four images (NIfTI or DICOM folders)
2. segment organs and muscles with **TotalSegmentator MRI** (no training needed)
3. per structure: **volume** and **proton-density fat fraction proxy** $FF = F / (F + W)$ (magnitude-based; no T2* or T1 correction)
4. write one row per participant to a CSV and **merge with your epidemiological variables**

> **Demo mode:** if you don't have Dixon data at hand, the cell below writes a small "NAKO-like" folder built from CHAOS (fat and water estimated from IP/OP), so you can test the pipeline first.

In [ ]:
DEMO_MODE = True                                   # set False and point NAKO_ROOT to your data
NAKO_ROOT = f"{WORK}/nako_like_demo" if DEMO_MODE else "/path/to/your/nako/dixon"
# file patterns per participant folder; adapt to the naming of your data export
PATTERNS = {"in": "*_in.nii.gz", "opp": "*_opp.nii.gz", "fat": "*_fat.nii.gz", "water": "*_water.nii.gz"}

if DEMO_MODE:
    for c in cases[:4]:
        d = f"{NAKO_ROOT}/sub-{int(c['id']):03d}" if str(c["id"]).isdigit() else f"{NAKO_ROOT}/sub-{c['id']}"
        os.makedirs(d, exist_ok=True)
        w, f = (c["ip"] + c["op"]) / 2, np.clip((c["ip"] - c["op"]) / 2, 0, None)
        for key, vol in {"in": c["ip"], "opp": c["op"], "fat": f, "water": w}.items():
            sitk.WriteImage(to_sitk(vol.astype(np.float32), c), f"{d}/{os.path.basename(d)}_{key}.nii.gz")
print(sorted(os.listdir(NAKO_ROOT))[:5], "...")

In [ ]:
from totalsegmentator.python_api import totalsegmentator
from totalsegmentator.map_to_binary import class_map
from scipy.ndimage import binary_erosion

STRUCTURES = ["liver", "spleen", "kidney_left", "kidney_right", "pancreas",
              "autochthon_left", "autochthon_right", "iliopsoas_left", "iliopsoas_right",
              "gluteus_maximus_left", "gluteus_maximus_right"]
SEG_INPUT = "in"            # which contrast to segment; TotalSegmentator MRI is sequence-independent, check a few cases visually
name2id = {v: k for k, v in class_map["total_mr"].items()}

def read(path_or_dir):
    if os.path.isdir(path_or_dir):                              # DICOM series folder
        r = sitk.ImageSeriesReader(); r.SetFileNames(r.GetGDCMSeriesFileNames(path_or_dir)); return r.Execute()
    return sitk.ReadImage(path_or_dir)

def process_participant(folder):
    files = {k: (glob.glob(os.path.join(folder, p)) or [None])[0] for k, p in PATTERNS.items()}
    if None in files.values():
        return {"id": os.path.basename(folder), "error": f"missing {[k for k, v in files.items() if v is None]}"}
    seg_path = os.path.join(folder, "totalseg_mr.nii.gz")
    if not os.path.exists(seg_path):
        totalsegmentator(files[SEG_INPUT], seg_path, task="total_mr", ml=True, quiet=True, fast=(DEVICE == "cpu"))
    seg_img = read(seg_path); seg = sitk.GetArrayFromImage(seg_img)
    fat = sitk.GetArrayFromImage(read(files["fat"])).astype(np.float32)
    water = sitk.GetArrayFromImage(read(files["water"])).astype(np.float32)
    ff = fat / (fat + water + 1e-6) * 100
    vox_ml = np.prod(seg_img.GetSpacing()) / 1000
    row = {"id": os.path.basename(folder)}
    for s in STRUCTURES:
        m = seg == name2id[s]
        row[f"{s}_vol_ml"] = m.sum() * vox_ml
        core = binary_erosion(m, iterations=2)
        row[f"{s}_ff_pct"] = float(np.median(ff[core])) if core.sum() > 50 else np.nan
    return row

folders = sorted(d for d in glob.glob(f"{NAKO_ROOT}/*") if os.path.isdir(d))
pheno = pd.DataFrame([process_participant(f) for f in folders])
pheno.to_csv(f"{WORK}/image_derived_phenotypes.csv", index=False)
pheno.round(1)

### Quality control first!
Before any statistics, look at the segmentations (e.g. a random sample of 50 participants, plus all outliers in volume or fat fraction). Automated QC ideas: volumes outside physiological ranges, structures touching the field-of-view border, fat–water swaps (FF > 50 % in liver or muscle).

In [ ]:
folder = folders[0]
img = sitk.GetArrayFromImage(read(glob.glob(os.path.join(folder, PATTERNS[SEG_INPUT]))[0]))
seg = sitk.GetArrayFromImage(read(os.path.join(folder, "totalseg_mr.nii.gz")))
z = int(np.argmax((seg == name2id["liver"]).sum((1, 2))))
plt.figure(figsize=(6, 6)); plt.imshow(img[z], cmap="gray"); plt.imshow(np.ma.masked_equal(seg[z], 0), cmap="tab20", alpha=0.5)
plt.title(os.path.basename(folder)); plt.axis("off"); plt.show()

### Merge with your cohort data and analyse
Replace the simulated covariates with your NAKO variables (merge on your participant ID).

In [ ]:
import statsmodels.formula.api as smf
rng = np.random.default_rng(0)
cov = pd.DataFrame({"id": pheno["id"], "age": rng.uniform(30, 70, len(pheno)),
                    "sex": rng.choice(["female", "male"], len(pheno)), "bmi": rng.normal(27, 4, len(pheno))})
# cov = pd.read_csv("my_nako_variables.csv")      # <- your data
df = pheno.merge(cov, on="id")
if len(df) > 10:
    print(smf.ols("liver_ff_pct ~ age + C(sex) + bmi", data=df).fit().summary())
else:
    print("Only", len(df), "participants in demo mode: run on your full data set to fit the model.")
    display(df.round(1))

**Remember from this week:** image-derived phenotypes are *measurements with error*. The segmentation quality may depend on BMI, age or sex (differential measurement error), and a regression of liver fat on BMI is not a causal effect without a DAG. 😉